In [ ]:
%matplotlib inline
import sys
import os
sys.path.append(os.path.abspath(".."))
import importlib

from functions import color_palettes
importlib.reload(color_palettes)
from functions.color_palettes import *

from functions import utils
importlib.reload(utils)
from functions.utils import *

from functions import config
importlib.reload(config)
from functions.config import *

sns.set_theme(style="whitegrid", context="talk")

c:\Users\aoezek\AppData\Local\miniconda3\envs\circadian\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
cluster_map = pd.read_excel(cluster_map_path)

age_gender_map=pd.read_excel(age_gender_map_path)
age_gender_map=age_gender_map.rename(columns={'ID': 'Participant'})

tuned_engineered_df=pd.read_excel(iauc_prediction_data_path)
participant_ids = tuned_engineered_df["Participant_id"].unique()

tuned_engineered_df_notfiltered=pd.read_excel(iauc_prediction_data_filtered_path)

In [ ]:
#top 20 features which have the highest correlation with iAUC
corr_df = tuned_engineered_df_notfiltered[meal_cols + meta_cols + glucose_cols + circadian_cols + ['iauc']]
corr = corr_df.corr()
highly_correlated = (
    corr['iauc']
    .abs()
    .sort_values(ascending=False)
    .drop('iauc')
    .head(20)
    .index
    .tolist()
)

In [ ]:
feature_subset_columns = {"Meal":meal_cols, 
                      "Meal+Meta":meal_cols + meta_cols,
                      "Meal+Meta+Glucose":meal_cols + meta_cols + glucose_cols, 
                      "Meal+Meta+Glucose+Circadian":meal_cols + meta_cols + glucose_cols + circadian_cols, 
                      "Meal+Circadian":meal_cols + circadian_cols, 
                      "Meal+Glucose":meal_cols + glucose_cols,
                      "Glucose+Meal Time":glucose_plus_time,
                      "Highly_correlated_20":highly_correlated
                     }

In [ ]:
tuned_feature_sets_with_labels, tuned_engineered_df = build_feature_sets_task1(tuned_engineered_df, feature_subset_columns)
tuned_feature_sets_with_labels_NF, tuned_engineered_df_notfiltered = build_feature_sets_task1(tuned_engineered_df_notfiltered, feature_subset_columns)

tuned_feature_sets = list(tuned_feature_sets_with_labels.values())
tuned_feature_sets_NF = list(tuned_feature_sets_with_labels_NF.values())

In [11]:
target = "iauc"
participant_col = "Participant_id"

---
#### GLOBAL MODEL (KFold)
---

In [ ]:
global_results = []

for subset_name, X_subset in tuned_feature_sets_with_labels_NF.items():

    X = X_subset.copy()
    y = tuned_engineered_df_notfiltered[target]
    subset_results = global_model(
        X=X,
        y=y,
        subset_name=subset_name,
        n_iter=20,
        cv_splits=5,
        cv_repeats=1
    )

    global_results.append(subset_results)

global_results_all= pd.concat(global_results, ignore_index=True)

In [ ]:
#with the full feature subset passed, global_model_with_selection function first runs a Random Forest and 
#chooses the 20 most important features, then runs the other models on these features
X = tuned_engineered_df_notfiltered[meal_cols +glucose_cols + circadian_cols +meta_cols]
y = tuned_engineered_df_notfiltered[target]

global_results_with_selection_all = global_model_with_selection(
        X=X,
        y=y,
        cv_repeats=1
    )

In [ ]:
global_all=pd.concat([global_results_all, global_results_with_selection_all],axis=0, ignore_index=True)
global_all.to_excel('global_all_1905.xlsx',index=False)
global_all[["Subset","Model","Best_R2","Std_R2", "RMSE", "MAE"]].to_latex( #for the thesis tex
    "global_results_all_1905.tex",
    index=False,
    float_format="%.3f"
)

In [ ]:
#choose the best algorithm for each subset
best_per_subset_all = (
    global_all
    .sort_values("Best_R2", ascending=False)
    .groupby("Subset")
    .first()
)
print(best_per_subset_all)

best_per_subset_df_all= pd.DataFrame(best_per_subset_all)
best_model_subset=best_per_subset_df_all.reset_index().round(3).sort_values(['Best_R2'],ascending=False).drop(columns=['Best_Params'])
best_model_subset.to_latex(
    "task1_global_model_best_table_1905.tex",
    index=False,
    caption="Performances of best models for each feature subset in the global model",
    label="tab:model1_1_results",
    float_format="%.3f"
)

In [ ]:
selected_20_features_all = global_results_with_selection_all.iloc[0]["Selected_Features"]
# output (ordered):
#'Baseline_Glucose', 'carbs_x_glucoseSD', 'cos_time', 'meal_dist_to_circadian', 'sin_time', 'carbs_x_baseline', 'fiber_g', 'protein_g', 'fat_g', 'carbs_x_acrophase', 'glycemic_index', 'carbs_g', 'SD_Speed', 'total_calories', 'mesor', 'Resting_heart_rate', 'mean', 'min', 'period', 'lt_3.9' 

In [ ]:
row = global_results_with_selection_all.iloc[0]   #all rows have the features and importances the same
features = row['Selected_Features']
importances = row['Feature_Importances']
importances_series = pd.Series(importances, index=features).sort_values(ascending=True)

plt.figure(figsize=(12, 6))
plt.barh(
    importances_series.index,
    importances_series.values,
    color=blue_palette[3]
)
plt.bar_label(
    plt.gca().containers[0],
    fmt='%.2f',
    fontsize=11,
    padding=3
)
plt.xlabel("Relative Importance")
plt.ylabel("Feature Name")
plt.title("Global Model Feature Importance Selected 20", fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=10)
plt.tight_layout()

plt.savefig('C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\feature_importance_global_1905.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(14,6))
barplot = sns.barplot(
    data=global_all,
    x="Subset",
    y="RMSE", #or MAE or R2
    hue="Model",
    ci=None,
    palette=mixed_palette,
    edgecolor="white"
)

#to put the value on the bar, later discarded
# for p in barplot.patches:
#     height = p.get_height()
#     if height > 0:   # skip empty bars
#         barplot.annotate(
#             f'{height:.3f}',
#             (p.get_x() + p.get_width()/2., height),
#             ha='center',
#             va='bottom',
#             fontsize=6,
#             rotation=45
#         )

plt.title("Global Model Performance (Mean RMSE) per Subset") #change if MAE or R2 plotted
plt.ylabel("Mean RMSE") #change if MAE or R2 plotted
plt.xlabel("Feature Subset")
plt.xticks(rotation=30, fontsize=10)
plt.yticks(fontsize=12)
plt.legend(bbox_to_anchor=(1, 1), loc=2, fontsize=10)
plt.tight_layout()
plt.savefig('C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\global_results_all_rmse_08.07.png', dpi=400, bbox_inches='tight')
plt.show()

---
#### Per Person Models with pre-defined feature subsets
- these models are trained separately on each participant
- however only the feature importance results are meaningful, as these models run the overfitting risk a lot and are nor reliable on such small training batches
###### (tuned_engineered_df has the 44 participants with <10 total photos excluded)
---

In [ ]:
# First plot of number of recorded meals (per participant dataset size) for each participant

meal_number_plot = (
    tuned_engineered_df.groupby('Participant_id')
    .size()
    .reset_index(name='Number of Meals')
    .rename(columns={'Participant_id': 'Participant'})
    .sort_values('Number of Meals', ascending=False)
)

meal_number_plot = meal_number_plot.merge(age_gender_map, on='Participant', how='left')
plt.figure(figsize=(14,8))

ax = sns.barplot(
    data=meal_number_plot,
    x="Participant",
    y="Number of Meals",
    hue='age_group_label',
    palette=[mixed_palette[2], mixed_palette[1]],
    alpha=0.8
)

for p in ax.patches:
    height = p.get_height()
    ax.annotate(
        f"{height:.0f}",
        (p.get_x() + p.get_width()/2, height),
        ha="center",
        va="bottom",
        fontsize=12
    )

plt.xticks(rotation=45, fontsize=12)
plt.ylabel("Number of Meal Images")
plt.xlabel("Participant")
plt.title("Number of Available Meal Images per Participant")
plt.legend()
plt.tight_layout()
plt.savefig('C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\number_of_meals_1905_by_agegroup.png', dpi=400)
plt.show()

In [ ]:
#model run wuth all pre-defined feature subsets
personal_results_all = []

for subset_name, features in feature_subset_columns.items():

    print(f"\nRunning subset: {subset_name}")

    X_subset = tuned_engineered_df[features + ["Participant_id"]]
    y = tuned_engineered_df[target]

    res = personalized_models(
        X=X_subset,
        y=y,
        participant_ids=participant_ids,
        subset_name=subset_name
    )

    personal_results_all.append(res)

personal_results_without_feature_selection = pd.concat(personal_results_all, ignore_index=True)
best_personal_results_without_feature_selection = personal_results_without_feature_selection.loc[personal_results_without_feature_selection.groupby("Participant")["Best_R2"].idxmax()].reset_index(drop=True).sort_values(['Best_R2'],ascending=False)

In [ ]:
latex_table_personal = best_personal_results_without_feature_selection.to_latex(
    "task1_personal_model_best_table_1905.tex",
    index=False,
    caption="Performances of best model-algorithm pairs for each participant",
    label="tab:personal_without_selection_full_results",
    float_format="%.3f"
)

In [ ]:
#which model performed better on what percent of the participants
model_pct = (best_personal_results_without_feature_selection["Model"].value_counts(normalize=True) * 100).round(2)
model_df = model_pct.reset_index()
model_df.columns = ["Model", "Percentage"]

plt.figure(figsize=(12, 6))

barplot = sns.barplot(
    data=model_df,
    x="Percentage",
    y="Model",
    color=blue_palette[3],
    edgecolor="white"
)

barplot.bar_label(barplot.containers[0], fmt="%.1f%%", fontsize=14)

barplot.set_xticks(np.arange(0, model_df["Percentage"].max() + 5, 5))

plt.title("Algorithm Selection Percentages", fontsize=16)
plt.ylabel("Algorithm")
plt.xlabel("Selection Percentage (%)")
plt.xticks(fontsize=12)
plt.yticks(fontsize=13)

plt.tight_layout()
#plt.savefig('/Users/aylinozek/Desktop/Thesis/Thesis_Analysis/visualisations/algorithm_selection_personalized.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
#which subset performed better on what percent of the participants
subset_pct = (best_personal_results_without_feature_selection["Subset"].value_counts(normalize=True) * 100).round(2)
subset_df = subset_pct.reset_index()
subset_df.columns = ["Subset", "Percentage"]

plt.figure(figsize=(12, 6))

barplot = sns.barplot(
    data=subset_df,
    x="Percentage",
    y="Subset",
    color=blue_palette[3],
    edgecolor="white"
)

barplot.bar_label(barplot.containers[0], fmt="%.1f%%", fontsize=14)
barplot.set_xticks(np.arange(0, subset_df["Percentage"].max() + 5, 5))
plt.title("Subset Selection Percentages", fontsize=16)
plt.ylabel("Subset")
plt.xlabel("Selection Percentage (%)")
plt.xticks(fontsize=12)
plt.yticks(fontsize=13)

plt.tight_layout()
#plt.savefig('/Users/aylinozek/Desktop/Thesis/Thesis_Analysis/visualisations/subset_selection_personalized.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
personal_results_without_feature_selection.to_excel('results\\excels\\personal_results_without_selection_1905.xlsx', index=False)

In [ ]:
temp = best_personal_results_without_feature_selection.merge(
    age_gender_map[["Participant", "age_group_label"]],
    on="Participant",
    how="left"
)
best_personal_results_without_feature_selection_young = temp[temp["age_group_label"] == "Age group 19-32"]
best_personal_results_without_feature_selection_old = temp[temp["age_group_label"] == "Age group 59-76"]

In [ ]:
#ALL per person model (with pre-defined feature subsets) plots with all evaluation metrics and separate for age groups

age_groups = {
    "young": best_personal_results_without_feature_selection_young,
    "old": best_personal_results_without_feature_selection_old,
}

metrics = {
    "Best_R2": {"label": "Best R²","higher_is_better": True, "filter": lambda df, col: df[col] > 0},
    "MAE":{"label": "Best MAE", "higher_is_better": False, "filter": lambda df, col: df[col].notna()},
    "RMSE": {"label": "Best RMSE", "higher_is_better": False, "filter": lambda df, col: df[col].notna()},
}

age_labels = {
    "young": "19-32",
    "old": "59-76",
}

for age_key, df_raw in age_groups.items():
    for metric_col, meta in metrics.items():

        df_plot = df_raw[meta["filter"](df_raw, metric_col)]
        df_plot = df_plot.sort_values(metric_col, ascending=not meta["higher_is_better"])

        plt.figure(figsize=(10, 6))

        ax = sns.barplot(
            data=df_plot,
            x="Participant",
            y=metric_col,
            hue="Subset",
            palette=mixed_palette,
        )

        plt.xticks(rotation=45, fontsize=14)
        plt.yticks(fontsize=18)
        ymax = df_plot[metric_col].max()
        ax.set_ylim(top=ymax * 1.25)
        plt.ylabel(meta["label"])
        plt.xlabel("Participant")
        plt.title(
            f"Personal Model Performance per Participant - {meta['label']}"
            f" Age Group {age_labels[age_key]}"
        )
        #as this is descending the legend does not look good
        if metric_col=='Best_R2':
            ax.legend(title=None, fontsize=14, loc="upper right")
        else:
            ax.legend(title=None, fontsize=12, loc="upper left")

        plt.tight_layout()
        save_path = (
            f"C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\best_subset_{metric_col}_{age_key}_1506_loop2.png")
        plt.savefig(save_path,bbox_inches="tight", dpi=400)
        plt.show()
        plt.close()

---
#### Per Person Models with best 10 features per participant
- same as above but this time 10 most important feature were selected for each prticipant andmodels trained on this 10-feature subset
---

In [ ]:
X = tuned_engineered_df[meal_cols +glucose_cols + circadian_cols +meta_cols + ['Participant_id']]
y = tuned_engineered_df[target]

personalized_results_with_selection = personalized_models_with_selection(
    X=X,
    y=y,
    participant_ids=participant_ids,
    n_top_features=10, 
    n_iter=20,
    cv_splits=5
)

best_personal_models_with_selection = personalized_results_with_selection.loc[personalized_results_with_selection.groupby("Participant")["Best_R2(Mean)"].idxmax()].reset_index(drop=True).sort_values(['Best_R2(Mean)'],ascending=False)

In [ ]:
#most selected features and their selection percentages, age stratifies
def parse_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return [i.strip() for i in x.strip("[]").split(",")]
    return x

most_selected_feaures = best_personal_models_with_selection.copy()
most_selected_feaures['Selected_Features_Cleaned'] = most_selected_feaures['Selected_Features'].apply(parse_list)

most_selected_feaures = most_selected_feaures.merge(
    age_gender_map[["Participant", "age_group"]],
    on="Participant",
    how="left"
)
freq_list = []

for group in ["19-32", "59-76"]:
    subset = most_selected_feaures[most_selected_feaures["age_group"] == group]
    
    all_features = [
        feature
        for sublist in subset["Selected_Features_Cleaned"]
        for feature in sublist
    ]
    
    counts = Counter(all_features)
    temp_df = pd.DataFrame(counts.items(), columns=["Feature", "Frequency"])
    temp_df["age_group"] = group
    
    freq_list.append(temp_df)

freq_df_age = pd.concat(freq_list)

# Normalizing (as the group sizes differ)
group_sizes = most_selected_feaures["age_group"].value_counts()
freq_df_age["Normalized_Frequency"] = freq_df_age.apply(
    lambda row: row["Frequency"] / group_sizes[row["age_group"]],
    axis=1
)

In [ ]:
#Plot feature selection percentages over all participants, age stratified
plt.figure(figsize=(12, 8))
ax = sns.barplot(
    data=freq_df_age,
    y="Feature",
    x="Normalized_Frequency",
    hue="age_group",
    palette=[mixed_palette[2], mixed_palette[5]],
    edgecolor="white"
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=11)

plt.title("Feature Selection Frequency (Age-Stratified)", fontsize=16)
plt.xlabel("Normalized Frequency")
plt.ylabel("Feature")
handles, labels = ax.get_legend_handles_labels()
new_labels = ["Age group 19-32", "Age group 59-76"]
ax.legend(handles, new_labels, title="Age Group")

plt.tight_layout()
#plt.savefig('C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\feature_frequencies_age_stratified_1905.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
best_personal_models_with_selection.to_excel('results\\excels\\top_10_features_per_part_1905.xlsx', index=False)
best_personal_models_with_selection.to_latex(
    "task1_personal_model_best_table_selection_1905.tex",
    index=False,
    caption="Results of the importance selected features subset for each participant",
    label="tab:personal_with_selection_full_results",
    float_format="%.3f"
)

In [ ]:
#merging with age map for visualisations
temp = best_personal_models_with_selection.merge(
    age_gender_map[["Participant", "age_group_label"]],
    on="Participant",
    how="left"
)
best_personal_results_with_selection_young = temp[temp["age_group_label"] == "Age group 19-32"]
best_personal_results_with_selection_old   = temp[temp["age_group_label"] == "Age group 59-76"]

In [ ]:
#ALL per person model (with selected features) plots with all evaluation metrics and separate for age groups

age_groups = {
    "young": best_personal_results_with_selection_young,
    "old": best_personal_results_with_selection_old,
}
metrics = {
    "Best_R2(Mean)": {"label": "Best R²", "higher_is_better": True,"filter": lambda df, col: df[col] > 0},
    "MAE":{"label": "Best MAE", "higher_is_better": False, "filter": lambda df, col: df[col] > 0},
    "RMSE": {"label": "Best RMSE", "higher_is_better": False, "filter": lambda df, col: df[col] > 0},
}
age_labels = {
    "young": "19-32",
    "old":"59-76",
}
for age_key, df_raw in age_groups.items():
    for metric_col, meta in metrics.items():

        df_plot = meta["filter"](df_raw, metric_col).pipe(lambda m: df_raw[m])
        # descending for R2, ascending for error metrics
        df_plot = df_plot.sort_values(metric_col, ascending=not meta["higher_is_better"])

        plt.figure(figsize=(9, 6))
        ax = sns.barplot(
            data=df_plot,
            x="Participant",
            y=metric_col,
            hue="Model",
            palette=mixed_palette,
        )

        plt.xticks(rotation=45, fontsize=14)
        plt.yticks(fontsize=17)
        plt.ylabel(meta["label"])
        plt.xlabel("Participant")
        plt.title(
            f"Personal Model Performance with Best 10 Features "
            f"Age Group {age_labels[age_key]}"
        )
        plt.legend(fontsize=14)
        plt.tight_layout()
        save_path = (
            "C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\"
            f"best_model_{metric_col}_{age_key}_1506_loop.png"
        )
        plt.savefig(save_path, dpi=400)
        plt.show()
        plt.close()

---
#### GLOBAL MODEL with LOPO-CV and the FUSION APPROACH: RESIDUAL CORRECTION ON GLOBAL MODEL (with Leave One Out)
---

In [ ]:
#fusion model run with the best performing feature subset + algorithm from above: full set and XGBoost
#inside this function the global model results with LOPO-CV is also saved, (above model was KFold)
fusion_model_data=tuned_engineered_df[meal_cols + meta_cols + glucose_cols + circadian_cols + ['Participant_id']]
X = fusion_model_data.copy()
y = tuned_engineered_df[target]
fusion_results, correctors, global_on_personal_df = fusion_model_loo(
    X, y, participant_ids, subset_name="Meal+Meta+Glucose+Circadian", algorithm='XGBoost')

In [ ]:
age_stratified_global_loo=global_on_personal_df.merge(age_gender_map, on='Participant', how='left')

In [ ]:
age_stratified_global_loo_sorted = age_stratified_global_loo.sort_values("R2", ascending=False).reset_index(drop=True)

plt.figure(figsize=(14, 8))
ax = sns.barplot(
    x="Participant",
    y="R2",
    data=age_stratified_global_loo_sorted,
    hue="age_group_label",
    palette=[mixed_palette[1], mixed_palette[2]],
    alpha=0.8
)

plt.axhline(y=age_stratified_global_loo_sorted["R2"].mean(), color=mixed_palette[0], linestyle="--", linewidth=1.5,
            label=f"mean R2={age_stratified_global_loo_sorted['R2'].mean():.3f}")
plt.axhline(y=0, color="black", linewidth=1.5)
plt.ylim([-1.2, 1.2])
plt.xticks(rotation=45, fontsize=12)
plt.ylabel("R2")
plt.xlabel("Participant")
plt.title("Global Model R2 per Participant (LOPO-CV) - Age-Stratified")
plt.legend()

plt.tight_layout()
plt.savefig('C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\loo_global_bar_by_agegroup_1905.png', dpi=400)
plt.show()

In [ ]:
fusion_results_plot = fusion_results[fusion_results["Alpha_Blend"] > 0].sort_values('Alpha_Blend', ascending=False)
#fusion_results_plot = fusion_results.sort_values('Fusion_MAE', ascending=True)
fusion_results_plot = fusion_results_plot.merge(age_gender_map, on='Participant', how='left')

plt.figure(figsize=(12,6))

ax = sns.barplot(
    data=fusion_results_plot,
    x="Participant",
    y="Alpha_Blend",
    hue="age_group_label",
    palette=[mixed_palette[2], mixed_palette[1]],
    alpha=0.8
)

plt.xticks(rotation=45, fontsize=10)
plt.ylabel("Alpha_Blend")
plt.xlabel("Participant")
plt.title("Correction Magnitude Coefficient per Participant (Age-Stratified) - Fusion Model")
plt.legend()
plt.tight_layout()
plt.savefig('C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\fusion_loo_model_alpha_1905_byage.png', dpi=400)
plt.show()

In [ ]:
fusion_results_plot = fusion_results[fusion_results["Fusion_R2"] > 0].sort_values('Fusion_R2',ascending=False)
fusion_results_plot = fusion_results_plot.merge(age_gender_map, on='Participant', how='left')

plt.figure(figsize=(12,6))

ax = sns.barplot(
    data=fusion_results_plot,
    x="Participant",
    y="Fusion_R2",
    hue="age_group_label",
    palette=[mixed_palette[1], mixed_palette[2]],
    alpha=0.8
)

plt.xticks(rotation=45, fontsize=10)
plt.ylabel("R²")
plt.xlabel("Participant")
plt.title("Performance per Participant (R²) - Fusion Model")
plt.legend()
plt.tight_layout()
plt.savefig('C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\fusion_loo_model_r2_1905_byage.png', dpi=400)
plt.show()

In [ ]:
fusion_results.to_excel('results\\excels\\fusion_loo_results_1806.xlsx', index=False)
global_on_personal_df.to_excel('results\\excels\\globals_on_participants_1806.xlsx', index=False)


In [26]:
cols = ["Participant", "Global_R2", "Global_MAE", "Global_RMSE"]
global_on_personal_df[cols].to_latex(
    "global_loo_per_participant.tex",
    index=False,
    caption="Performance of the global model with LOPO-CV on each participant, reported as $R^2$, MAE, RMSE. On the full feature subset; Meal+Meta+Glucose+Circadian and using XGBoost as the winner algorithm",
    label="tab:global_on_participants_results",
    float_format="%.3f"
)

In [ ]:
global_on_personal_df.describe()

In [ ]:
fusion_results.describe()

---
#### COMPARISON:
- comparison of per participant performances of global, per person, and residual corrected global (fusion)
---

In [ ]:
comparison = compare_all_models(
    global_on_personal_df=global_on_personal_df,
    personal_results_df=personal_results_without_feature_selection,
    fusion_results_df=fusion_results,
    subset_name="Meal+Meta+Glucose+Circadian",
    best_personal_model='XGBoost'
)

In [ ]:
comparison_sorted = comparison.sort_values("Fusion_RMSE", ascending=True)

x = np.arange(len(comparison_sorted))
width = 0.35
fig, ax = plt.subplots(figsize=(16, 8))

ax.bar(x - width/2, comparison_sorted["Global_RMSE"], width, label="Global", color=mixed_palette[1])
ax.bar(x + width/2, comparison_sorted["Fusion_RMSE"], width, label="Fusion", color=mixed_palette[0])

ax.axhline(y=0, color="black", linestyle="-", linewidth=1.2)
ax.set_xticks(x)
ax.set_xticklabels(comparison_sorted["Participant"], rotation=45, fontsize=11)
ax.set_ylabel("RMSE")
ax.set_ylim(0,3)
ax.set_title("Global vs Fusion RMSE per Participant (LOPO-CV)")
ax.legend()
plt.tight_layout()
plt.savefig("C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\global_vs_fusion_1905_rmse.png", dpi=400)
plt.show()

In [ ]:
# for age stratified comparison visuals, visualisations below, not looped: change needed wrt to the plot wanted (old/young or r2, rmse, mae)
temp = comparison_sorted.merge(
    age_gender_map[["Participant", "age_group_label"]],
    on="Participant",
    how="left"
)
comparison_sorted_young = temp[temp["age_group_label"] == "Age group 19-32"]
comparison_sorted_old   = temp[temp["age_group_label"] == "Age group 59-76"]

In [ ]:
x = np.arange(len(comparison_sorted_old))
width = 0.35
fig, ax = plt.subplots(figsize=(16, 8))

ax.bar(x - width/2, comparison_sorted_old["Global_RMSE"], width, label="Global", color=mixed_palette[1])
ax.bar(x + width/2, comparison_sorted_old["Fusion_RMSE"], width, label="Fusion", color=mixed_palette[0])

ax.axhline(y=0, color="black", linestyle="-", linewidth=1.2)
ax.set_xticks(x)
ax.set_xticklabels(comparison_sorted_old["Participant"], rotation=45, fontsize=11)
ax.set_ylabel("RMSE")
ax.set_ylim(0,3)
ax.set_title("Global vs Fusion RMSE per Participant (LOPO-CV)- Age Group 59-76")
ax.legend()
plt.tight_layout()
plt.savefig("C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\global_vs_fusion_1905_rmse_old.png", dpi=400)
plt.show()

In [ ]:
x = np.arange(len(comparison_sorted_old))
width = 0.35

fig, ax = plt.subplots(figsize=(16, 8))

ax.bar(x - width/2, comparison_sorted_old["Global_R2"], width, label="Global", color=mixed_palette[1])
ax.bar(x + width/2, comparison_sorted_old["Fusion_R2"], width, label="Fusion", color=mixed_palette[0])

ax.axhline(y=0, color="black", linestyle="-", linewidth=1.2)
ax.set_xticks(x)
ax.set_xticklabels(comparison_sorted_old["Participant"], rotation=45, fontsize=11)
ax.set_ylabel("R²")
ax.set_ylim(-1.5,1.5)
ax.set_title("Global vs Fusion R² per Participant (LOPO-CV)- Age Group 59-76")
ax.legend()
plt.tight_layout()
plt.savefig("C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\global_vs_fusion_1905_old.png", dpi=400)
plt.show()

In [ ]:
heat_data = (
    comparison
    .sort_values("Fusion_R2", ascending=False)
    .set_index("Participant")[["Global_R2", "Personal_R2", "Fusion_R2"]]
    .rename(columns={
        "Global_R2": "Global",
        "Personal_R2":"Personal",
        "Fusion_R2": "Fusion"
    })
)

fig, ax = plt.subplots(figsize=(8, 16))

sns.heatmap(
    heat_data,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor="gray",
    ax=ax,
    annot_kws={"size": 8, "color": "#161616"},
    cbar_kws={"shrink": 0.3, 'aspect':10, "label": "R²"}
)

ax.set_title("R² per Participant - Global vs Personal vs Fusion", pad=16,fontsize=15)
ax.set_xlabel("")
ax.set_ylabel("Participant",fontsize=12)
ax.tick_params(axis="x", labelsize=10)
ax.tick_params(axis="y", labelsize=8, rotation=0)
ax.collections[0].colorbar.ax.tick_params(labelsize=7)
ax.collections[0].colorbar.set_label("R²", fontsize=9)

plt.tight_layout()
plt.savefig("comparison_heatmap_1905.png", dpi=400, bbox_inches="tight")
plt.show()

---
#### RESIDUAL CORRECTION FUSION MODEL ON CLUSTERS
---

In [ ]:
# clustered ids: 39 (further elimination wrt fasting times and avg meals per day)
clustering_ids = [
    'P16', 'P03', 'P23', 'P32', 'P24', 'P69', 'P35', 'P07', 'P22', 
    'P25', 'P15', 'P47', 'P04', 'P26', 'P36', 'P13', 'P38', 'P37', 
    'P57', 'P17', 'P06', 'P14', 'P21', 'P40', 'P11', 'P46', 'P30', 
    'P05', 'P53', 'P02', 'P09', 'P08', 'P20', 'P48', 'P01', 'P45', 
    'P51', 'P33', 'P60'
]
tuned_engineered_df_clustering = tuned_engineered_df[tuned_engineered_df['Participant_id'].isin(clustering_ids)].copy()

##### merge with 2 Response Clusters by PPGR METRICS

In [ ]:
data_with_clusters = tuned_engineered_df_clustering.merge(
    cluster_map, on='Participant_id'
)

In [ ]:
# running the above residual correction fusion approach with the same architecture, only difference 
#is global means on cluster population
correctors_per_cluster = []
global_on_personal_per_cluster = []
results_per_cluster=[]

for cluster_id, df_cluster in data_with_clusters.groupby('cluster'):
    
    print(f"Running model for Cluster {cluster_id}")
    
    fusion_model_data = df_cluster[meal_cols + meta_cols + glucose_cols + circadian_cols + ['Participant_id']]
    X = fusion_model_data.copy()
    y = df_cluster[target]
    
    participant_ids_cluster = df_cluster['Participant_id'].unique()
    
    fusion_cluster_results, correctors_cluster, global_on_personal_df_cluster = fusion_model_loo(
        X, y, participant_ids_cluster,
        subset_name=f"Cluster_{cluster_id}",
        algorithm='XGBoost'
    )
    
    fusion_cluster_results['cluster'] = cluster_id
    results_per_cluster.append(fusion_cluster_results)
    correctors_per_cluster.append(correctors_cluster)
    global_on_personal_per_cluster.append(global_on_personal_df_cluster)
    
results_per_cluster_all = pd.concat(results_per_cluster, ignore_index=True)
global_on_personal_per_cluster_all = pd.concat(global_on_personal_per_cluster, ignore_index=True)
correctors_per_cluster_all = pd.concat([pd.DataFrame(c).T for c in correctors_per_cluster], ignore_index=True)

In [ ]:
#clustering_fusion_results_plot = results_per_cluster_all[results_per_cluster_all["Fusion_R2"] > 0].sort_values('Fusion_R2',ascending=False)
clustering_fusion_results_plot = results_per_cluster_all.sort_values('Fusion_MAE',ascending=True)

plt.figure(figsize=(12,6))

ax = sns.barplot(
    data=clustering_fusion_results_plot,
    x="Participant",
    y="Fusion_MAE",
    palette=[mixed_palette[1],  mixed_palette[2]],
    hue='Subset'
)
plt.xticks(rotation=45, fontsize=10)
plt.ylabel("MAE")
plt.xlabel("Participant")
plt.title("Performance per Participant - Fusion Model on Clusters")
plt.tight_layout()
plt.savefig('C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\fusion_loo_model_cluster_mae_1905.png', dpi=400)
plt.show()

In [ ]:
results_per_cluster_all.groupby('cluster').mean(numeric_only=True)